---
<div align="center">

# **MÉTODOS DE IMPUTACIÓN**

</div>

---

<div style="display: flex; align-items: center;">

  <!-- Columna texto -->
  <div style="width: 50%; padding-right: 20px;">
    <p style="text-align: justify;">
      Los métodos de imputación son técnicas usadas en Analítica de Datos para reemplazar valores faltantes (missing values) en un conjunto de datos con estimaciones razonables.
     </p>
  </div>

  <!-- Columna imagen -->
  <div style="width: 50%; text-align: center;">
    <img src="img/missi.png" width="300" height="200">
  </div>

</div>








---
# Imputación Numérica
---
### 1. Imputación por Medida de Tendencia Central

La imputación por medida de tendencia central es uno de los métodos más simples para tratar datos faltantes: consiste en reemplazar los valores faltantes por un valor representativo de la variable, típicamente:

- Media 

- Mediana

- Moda


---
### 2. Imputación por KNN

La imputación por KNN (K-Nearest Neighbors) es un método que rellena valores faltantes usando la información de observaciones similares en el dataset. Para imputar un dato faltante, el método busca las k observaciones más parecidas y usa sus valores para estimarlo.

“Dime quiénes son tus vecinos… y te diré cuánto vales.”


---
### 3. Iterativa Tipo MICE

La imputación iterativa (tipo MICE, Multiple Imputation by Chained Equations) es un método que rellena valores faltantes modelando cada variable como función de las demás, de manera iterativa.

“Cada variable con missing se predice usando el resto… y este proceso se repite hasta estabilizarse.”



---
### 4. Iterative + Random Forest

La imputación **Iterative + Random Forest** es una versión avanzada de la imputación iterativa (tipo MICE) donde, en lugar de usar modelos lineales, se utiliza un **Random Forest** para predecir los valores faltantes en cada paso.


“Imputas cada variable usando las demás… pero con un modelo potente que captura relaciones no lineales.”

---


# Imputación Categórica

---
### 1. Imputación Iterative + Clasificador 

La imputación Iterative + Clasificador es la versión para variables categóricas del enfoque tipo MICE, donde cada variable con valores faltantes se predice mediante un modelo de clasificación usando las demás variables como entrada.

“Si una variable es categórica, no la predigo con regresión… la clasifico.”

---
### 2. Imputación Miss Forest

La imputación categórica con MissForest es un método que usa **Random Forest** para rellenar valores faltantes en variables categóricas (y también numéricas), aprovechando las relaciones entre todas las variables del dataset.

“Predigo las categorías faltantes usando muchos árboles de decisión basados en el resto de variables.”

---

In [ ]:
# Librerías #
import numpy as np
import pandas as pd
import os

from sklearn.datasets import load_diabetes
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

## Funciones Pesonalizadas - Valores Perdidos y Nulos #

In [ ]:
# Porcentaje de valores perdidos por variable #
def resum_missing(df):
    total = df.isnull().sum().sort_values(ascending=False)
    percent = (df.isnull().sum() *100/df.isnull().count()).sort_values(ascending=False)
    missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Porcentaje'])
    return missing_data

# Resumen de posición de valores perdidos #
def posmissing(df):
    lista_miss = np.where(df.isna())
    v1 = pd.Series(np.ndarray.tolist(lista_miss[0]))
    v2 = pd.Series(np.ndarray.tolist(lista_miss[1]))
    resumen_vna = pd.concat([v1, v2], axis=1, keys=['posicion_fila', 'posicion_columna'])
    return resumen_vna

# Construcción función que cuenta ceros #
def count_zeros(df):
    total = (df == 0).astype(int).sum(axis=0)
    percent = ((df == 0).astype(int).sum(axis=0) *100/df.count())
    missing_data = pd.concat([total, percent], axis=1, keys=['Total_ceros', 'Porcentaje'])
    return missing_data.sort_values(by='Porcentaje', ascending=False)

## Experimento con Variables Numéricas

In [ ]:
# Cargar la Base de Datos Diabetes #
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df.head()

In [ ]:
# Introducir Valores Faltantes #
def introducir_missing(df, porcentaje=0.2):
    df_missing = df.copy()
    mask = np.random.rand(*df.shape) < porcentaje
    df_missing[mask] = np.nan
    return df_missing, mask

df_missing, mask = introducir_missing(df, 0.2)

In [ ]:
# Resumen de Datos Faltantes y Porcentajes #
resum_missing(df_missing)

In [ ]:
# Dimensión de los Datos #
df_missing.shape

In [ ]:
# 1. Tendencia Central
mean_imp = SimpleImputer(strategy='mean')
df_mean = mean_imp.fit_transform(df_missing)

# 2. KNN
knn_imp = KNNImputer(n_neighbors=5)
df_knn = knn_imp.fit_transform(df_missing)

# 3. Iterative (tipo MICE)
iter_imp = IterativeImputer(random_state=42)
df_iter = iter_imp.fit_transform(df_missing)

# 4. Iterative + Random Forest
rf_imp = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    random_state=42
)
df_rf = rf_imp.fit_transform(df_missing)

In [ ]:
# Medir Variables Imputadas con Variables Originales #
def evaluar(original, imputado, mask):
    return {
        "RMSE": np.sqrt(mean_squared_error(original[mask], imputado[mask])),
        "MAE": mean_absolute_error(original[mask], imputado[mask])
    }

resultados = pd.DataFrame({
    "Mean": evaluar(df.values, df_mean, mask),
    "KNN": evaluar(df.values, df_knn, mask),
    "Iterative": evaluar(df.values, df_iter, mask),
    "RF_Iterative": evaluar(df.values, df_rf, mask)
}).T

resultados

In [ ]:
# Visualización de Errores #
resultados.plot(kind='bar')
plt.title("Comparación de Métodos de Imputación")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.grid()
plt.show()

In [ ]:
# Lista de Variables de Diabetes #
df.columns

In [ ]:
# Comparando las densidades  #
col = df.columns[2] # Posición [0] corresponde a la edad #

plt.figure(figsize=(10,5))

sns.kdeplot(df[col], label="Original", linewidth=2)
sns.kdeplot(df_mean[:,0], label="Mean")
sns.kdeplot(df_knn[:,0], label="KNN")
sns.kdeplot(df_iter[:,0], label="Iterative")

plt.legend()
plt.title(f"Distribución - {col}")
plt.show()

---
# Imputación en Datos Reales

In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score

import matplotlib.pyplot as plt
import os

In [ ]:
# Organizar Rutas "os" #
mainpath= "/Users/duvancatano/Documents/Data_Analytics_UdeA/ml-project/data/fraud"
filename= "train.csv"
fullpath= os.path.join(mainpath,filename)

In [ ]:
# Cargar Datos #
data = pd.read_csv(fullpath, sep=";") # El separador es "," porque en el archivo .csv los valores están separados por coma
pd.set_option('display.max_columns', None) # Para mostrar todas las columnas del DataFrame sin truncar

In [ ]:
# Resumen de Datos Faltantes #
resum_missing(data)

In [ ]:
# Posición de Datos Faltantes #
posmissing(data)

In [ ]:
# Muestra Aleatoria de 20 registros #
data.sample(20)

In [ ]:
# Identificar Variables de Interés #
num_cols = data[['Dist_Mean_NAL', 'EGRESOS', 'INGRESOS', 'EDAD']].select_dtypes(include=np.number).columns
cat_cols = data[['SEXO', 'SEGMENTO']].select_dtypes(exclude=np.number).columns

print("Numéricas:", list(num_cols))
print("Categóricas:", list(cat_cols))

In [ ]:
# Ejecución de Imputación Numérica #
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

num_imp = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    random_state=42
)

data_num_imputed = data.copy()

if len(num_cols) > 0:
    data_num_imputed[num_cols] = num_imp.fit_transform(data[num_cols])

In [ ]:
# Resumen de Base Imputada Numéricas#
resum_missing(data_num_imputed)

In [ ]:
# Ejecución de Imputación Categórica # 

from sklearn.impute import SimpleImputer

cat_imp = SimpleImputer(strategy='most_frequent')

data_imputed = data_num_imputed.copy()

if len(cat_cols) > 0:
    data_imputed[cat_cols] = cat_imp.fit_transform(data[cat_cols])

In [ ]:
# Resumen de Base Imputada Categóricas #
resum_missing(data_imputed)

In [ ]:
# Ejemplo de Comparación de Densidades #

col = num_cols[3] #'Dist_Mean_NAL', 'EGRESOS', 'INGRESOS', 'EDAD'

plt.figure(figsize=(10,5))

sns.kdeplot(data[col], label="Original (con missing)", linewidth=2)
sns.kdeplot(data_imputed[col], label="Imputado")

plt.legend()
plt.title(f"Distribución: {col}")
plt.show()

---
# ¡FIN!
---